# Notebook 6: Advanced Topics — MoE, LoRA, and Alignment

**Goal:** Understand Mistral's unique architectural choices (MoE) and fine-tuning techniques (LoRA/DPO).

**Topics:**
1. Mixture of Experts (MoE) — routing mechanism from scratch
2. Load balancing in MoE
3. LoRA (Low-Rank Adaptation) — implement from scratch
4. QLoRA concept
5. DPO (Direct Preference Optimization) loss
6. Evaluation: perplexity, LLM-as-judge
7. Grouped Query Attention (GQA)
8. Sliding Window Attention (SWA)
9. RoPE (Rotary Position Embedding)
10. Practice: spot the architecture bug

In [ ]:
import numpy as np
import math
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass

np.random.seed(42)
print('Setup complete')

---
## 1. Mixture of Experts (MoE) — Core Concept

**Idea:** Replace the single FFN layer with N expert FFN layers. A router picks top-K experts for each token.

**Mixtral 8x7B specifics:**
- 8 expert FFN layers per Transformer block
- Router picks top-2 experts per token
- Only 2/8 experts are active → compute = ~13B active params (not 47B)
- All 47B params are loaded into GPU memory

**Why MoE?** Scales model capacity (more parameters, more 'knowledge') without proportionally scaling compute.

In [ ]:
class Expert:
    """A single FFN expert (simplified — no SwiGLU for clarity)."""
    def __init__(self, d_model: int, d_ff: int):
        scale = 0.02
        self.W1 = np.random.randn(d_model, d_ff) * scale
        self.W2 = np.random.randn(d_ff, d_model) * scale
    
    def forward(self, x: np.ndarray) -> np.ndarray:
        return np.maximum(0, x @ self.W1) @ self.W2  # ReLU FFN


class MoELayer:
    """
    Sparse Mixture of Experts layer.
    Replaces a single FFN in a Transformer block.
    """
    
    def __init__(self, d_model: int, d_ff: int, num_experts: int = 8, top_k: int = 2):
        self.num_experts = num_experts
        self.top_k = top_k
        self.d_model = d_model
        
        # Expert networks
        self.experts = [Expert(d_model, d_ff) for _ in range(num_experts)]
        
        # Router: linear layer mapping token -> expert logits
        self.router = np.random.randn(d_model, num_experts) * 0.02
    
    def route(self, x: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        """
        Compute routing probabilities for each token.
        
        x: (batch * seq, d_model) — flattened tokens
        Returns:
            top_k_weights: (batch*seq, top_k) — softmax weights for selected experts
            top_k_indices: (batch*seq, top_k) — which experts are selected
        """
        # YOUR CODE HERE
        # 1. Compute router logits: x @ self.router -> (tokens, num_experts)
        # 2. Softmax to get probabilities
        # 3. Select top-k experts per token
        # 4. Renormalize the top-k weights
        raise NotImplementedError()
    
    def forward(self, x: np.ndarray) -> np.ndarray:
        """
        x: (batch, seq, d_model)
        Returns: (batch, seq, d_model)
        """
        batch, seq, d = x.shape
        x_flat = x.reshape(-1, d)  # (batch*seq, d_model)
        
        # YOUR CODE HERE
        # 1. Get routing weights and expert indices
        # 2. For each token, compute weighted sum of top-k expert outputs
        # 3. Reshape back to (batch, seq, d_model)
        raise NotImplementedError()


# Test
np.random.seed(42)
moe = MoELayer(d_model=64, d_ff=256, num_experts=8, top_k=2)
x = np.random.randn(2, 4, 64)
# out = moe.forward(x)
# print('MoE output shape:', out.shape)  # (2, 4, 64)

In [ ]:
# REFERENCE
def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)


class MoELayerRef:
    def __init__(self, d_model: int, d_ff: int, num_experts: int = 8, top_k: int = 2):
        self.num_experts = num_experts
        self.top_k = top_k
        self.d_model = d_model
        self.experts = [Expert(d_model, d_ff) for _ in range(num_experts)]
        self.router = np.random.randn(d_model, num_experts) * 0.02
    
    def route(self, x: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        # x: (tokens, d_model)
        logits = x @ self.router  # (tokens, num_experts)
        probs = softmax(logits, axis=-1)
        
        # Top-k indices per token
        top_k_indices = np.argsort(probs, axis=-1)[:, -self.top_k:]  # (tokens, top_k)
        top_k_probs = np.take_along_axis(probs, top_k_indices, axis=-1)
        
        # Renormalize top-k weights
        top_k_weights = top_k_probs / top_k_probs.sum(axis=-1, keepdims=True)
        
        return top_k_weights, top_k_indices
    
    def forward(self, x: np.ndarray) -> np.ndarray:
        batch, seq, d = x.shape
        x_flat = x.reshape(-1, d)  # (tokens, d_model)
        tokens = x_flat.shape[0]
        
        weights, indices = self.route(x_flat)  # (tokens, top_k) each
        
        output = np.zeros_like(x_flat)
        
        for k in range(self.top_k):
            expert_ids = indices[:, k]  # (tokens,)
            expert_weights = weights[:, k:k+1]  # (tokens, 1)
            
            # Dispatch tokens to experts
            for expert_id in range(self.num_experts):
                token_mask = expert_ids == expert_id
                if token_mask.any():
                    expert_input = x_flat[token_mask]
                    expert_output = self.experts[expert_id].forward(expert_input)
                    output[token_mask] += expert_output * expert_weights[token_mask]
        
        return output.reshape(batch, seq, d)


MoELayer = MoELayerRef

np.random.seed(42)
moe = MoELayer(d_model=64, d_ff=256, num_experts=8, top_k=2)
x = np.random.randn(2, 4, 64)
out = moe.forward(x)
print('MoE output shape:', out.shape)

---
## 2. MoE Load Balancing

**Problem:** Without a balancing loss, the router collapses — it always picks the same 1-2 experts (they get more training, get better, router picks them more — feedback loop).

**Fix:** Add an auxiliary load balancing loss that encourages equal expert utilization.

In [ ]:
def load_balancing_loss(
    router_probs: np.ndarray,  # (tokens, num_experts)
    top_k_indices: np.ndarray,  # (tokens, top_k)
    num_experts: int,
) -> float:
    """
    Auxiliary load balancing loss (from Switch Transformer paper).
    
    L_aux = num_experts * sum_i(f_i * P_i)
    where:
    - f_i = fraction of tokens dispatched to expert i
    - P_i = mean router probability for expert i
    
    Perfect balance: L_aux = 1.0
    Collapsed: L_aux >> 1.0
    """
    # YOUR CODE HERE
    raise NotImplementedError()


# REFERENCE
def load_balancing_loss_ref(
    router_probs: np.ndarray,
    top_k_indices: np.ndarray,
    num_experts: int,
) -> float:
    tokens = router_probs.shape[0]
    
    # f_i: fraction of tokens routed to expert i
    f = np.zeros(num_experts)
    for idx in top_k_indices.flatten():
        f[idx] += 1
    f = f / (tokens * top_k_indices.shape[1])  # normalize by total expert assignments
    
    # P_i: mean router probability for expert i
    P = router_probs.mean(axis=0)  # (num_experts,)
    
    return float(num_experts * np.dot(f, P))


load_balancing_loss = load_balancing_loss_ref

# Test: balanced routing
np.random.seed(42)
tokens, experts, top_k = 100, 8, 2
uniform_probs = np.ones((tokens, experts)) / experts
balanced_indices = np.array([[i % experts, (i+1) % experts] for i in range(tokens)])
print('Balanced routing loss:', load_balancing_loss(uniform_probs, balanced_indices, experts))

# Collapsed routing (always expert 0)
collapsed_probs = np.zeros((tokens, experts))
collapsed_probs[:, 0] = 1.0
collapsed_indices = np.zeros((tokens, top_k), dtype=int)
print('Collapsed routing loss:', load_balancing_loss(collapsed_probs, collapsed_indices, experts))

---
## 3. LoRA — Low-Rank Adaptation

**Problem:** Full fine-tuning updates all 7B parameters — expensive, requires storing gradients for all params.

**LoRA idea:** Freeze the original weights W₀. Add a trainable low-rank decomposition:

```
W = W₀ + ΔW = W₀ + B @ A
```

- W₀ ∈ ℝ^(d×k) — frozen original weight
- A ∈ ℝ^(r×k) — trainable, random init
- B ∈ ℝ^(d×r) — trainable, zero init (so initially ΔW = 0)
- r << d, k — rank (typically 4-64)

**Parameter savings:** Instead of d×k params, only train d×r + r×k = r(d+k) params.
For d=k=4096, r=8: `8 * 8192 = 65536` vs `4096 * 4096 = 16.7M`

In [ ]:
class LoRALinear:
    """
    A linear layer with LoRA adapters.
    During inference: output = x @ (W0 + B @ A) * scale
    During training: only A and B are updated.
    """
    
    def __init__(
        self,
        in_features: int,
        out_features: int,
        rank: int = 8,
        alpha: float = 16.0,  # scaling factor; often = rank
    ):
        self.rank = rank
        self.alpha = alpha
        self.scale = alpha / rank  # scaling multiplier
        
        # Frozen base weight
        self.W0 = np.random.randn(in_features, out_features) * 0.02
        
        # Trainable LoRA matrices
        # YOUR CODE HERE: initialize A and B
        # A: random (kaiming), shape (rank, in_features) — NOTE: A is (r, in_feat) in most impls
        # B: zeros, shape (out_features, rank)
        # Wait — standard: A is (in_features, r), B is (r, out_features)
        # So W + B @ A gives (out_feat, in_feat) + (out_feat, r) @ (r, in_feat)... let's use:
        # Output = x @ W0 + x @ A @ B where A: (in, r), B: (r, out)
        raise NotImplementedError()
    
    def forward(self, x: np.ndarray) -> np.ndarray:
        """
        x: (..., in_features)
        Returns: (..., out_features)
        """
        # YOUR CODE HERE
        # base_out = x @ W0
        # lora_out = x @ A @ B * scale
        # return base_out + lora_out
        raise NotImplementedError()
    
    def trainable_params(self) -> int:
        """Count trainable parameters (A and B only)."""
        # YOUR CODE HERE
        raise NotImplementedError()
    
    def total_params(self) -> int:
        """Count all parameters including frozen W0."""
        # YOUR CODE HERE
        raise NotImplementedError()

# Test
# lora = LoRALinear(4096, 4096, rank=8, alpha=16)
# x = np.random.randn(2, 10, 4096)
# out = lora.forward(x)
# print('LoRA output shape:', out.shape)
# print(f'Trainable: {lora.trainable_params():,} / {lora.total_params():,} ({lora.trainable_params()/lora.total_params():.2%})')

In [ ]:
# REFERENCE
class LoRALinearRef:
    def __init__(self, in_features: int, out_features: int, rank: int = 8, alpha: float = 16.0):
        self.rank = rank
        self.alpha = alpha
        self.scale = alpha / rank
        
        # Frozen base weight
        self.W0 = np.random.randn(in_features, out_features) * 0.02
        
        # Trainable LoRA: x @ A @ B
        # A: random init (kaiming), B: zero init (so initial output = base only)
        std = 1.0 / math.sqrt(in_features)
        self.A = np.random.randn(in_features, rank) * std  # (in_feat, r)
        self.B = np.zeros((rank, out_features))              # (r, out_feat)
    
    def forward(self, x: np.ndarray) -> np.ndarray:
        base_out = x @ self.W0
        lora_out = (x @ self.A) @ self.B  # (batch, seq, r) @ (r, out_feat)
        return base_out + self.scale * lora_out
    
    def trainable_params(self) -> int:
        return self.A.size + self.B.size
    
    def total_params(self) -> int:
        return self.W0.size + self.A.size + self.B.size
    
    def merge_weights(self) -> np.ndarray:
        """Merge LoRA into base weights for inference (no overhead)."""
        return self.W0 + self.scale * (self.A @ self.B)


LoRALinear = LoRALinearRef

np.random.seed(42)
lora = LoRALinear(4096, 4096, rank=8, alpha=16)
x = np.random.randn(2, 10, 4096)
out = lora.forward(x)
print('LoRA output shape:', out.shape)
print(f'Trainable params: {lora.trainable_params():,}')
print(f'Total params:     {lora.total_params():,}')
print(f'Trainable ratio:  {lora.trainable_params()/lora.total_params():.4%}')

# Verify that at init, LoRA output equals base output (B is zeros)
base_out = x @ lora.W0
print(f'\nAt init, LoRA == base: {np.allclose(out, base_out)}')

---
## 4. Where to Inject LoRA Adapters

Original LoRA paper: inject into Q, V projection matrices.
Modern practice: inject into Q, K, V, O, and sometimes FFN projections.

**Interview Q:** *Why V and not just Q? Why inject into attention projections rather than FFN?*

Answer: Q and K together control *what* information is attended to. V controls *what* is extracted. Both matter. FFN is often sufficient for task-specific knowledge, but for instruction following, attention adapters help.

In [ ]:
def count_lora_params(
    d_model: int = 4096,
    num_layers: int = 32,
    rank: int = 8,
    target_modules: List[str] = ['q', 'v'],
) -> Dict:
    """
    Calculate LoRA parameter count for a full model.
    """
    # Each projection: in=d_model, out=d_model, so LoRA params = 2 * d_model * rank
    params_per_module = 2 * d_model * rank  # A + B
    total_lora = len(target_modules) * num_layers * params_per_module
    total_base = 7_300_000_000  # Mistral 7B
    
    return {
        'lora_params': total_lora,
        'base_params': total_base,
        'ratio': total_lora / total_base,
        'params_per_layer': len(target_modules) * params_per_module,
    }


print('LoRA parameter counts for Mistral 7B:')
for modules, rank in [(['q', 'v'], 8), (['q', 'k', 'v', 'o'], 8), (['q', 'k', 'v', 'o'], 64)]:
    stats = count_lora_params(rank=rank, target_modules=modules)
    print(f"  modules={modules}, r={rank}: {stats['lora_params']/1e6:.1f}M params ({stats['ratio']:.3%} of base)")

---
## 5. DPO (Direct Preference Optimization) Loss

**RLHF problem:** Requires training a reward model AND running PPO — complex and unstable.

**DPO insight:** The optimal RLHF policy has a closed-form expression in terms of a reference model. We can directly optimize for it without a reward model.

**DPO loss:**
```
L_DPO = -log(sigmoid(β * (log π(y_w|x) - log π_ref(y_w|x)) - (log π(y_l|x) - log π_ref(y_l|x))))
```

Where:
- y_w = preferred (winning) response
- y_l = dispreferred (losing) response
- π = policy (model being trained)
- π_ref = reference model (frozen SFT model)
- β = temperature (controls how much we deviate from reference)

In [ ]:
def dpo_loss(
    log_prob_policy_win: float,    # log π(y_w|x)
    log_prob_policy_lose: float,   # log π(y_l|x)
    log_prob_ref_win: float,       # log π_ref(y_w|x)
    log_prob_ref_lose: float,      # log π_ref(y_l|x)
    beta: float = 0.1,
) -> float:
    """
    Compute DPO loss for a single preference pair.
    """
    # YOUR CODE HERE
    # reward_win = log_prob_policy_win - log_prob_ref_win  (implicit reward difference)
    # reward_lose = log_prob_policy_lose - log_prob_ref_lose
    # loss = -log(sigmoid(beta * (reward_win - reward_lose)))
    raise NotImplementedError()


def sigmoid(x: float) -> float:
    return 1.0 / (1.0 + math.exp(-x))


# REFERENCE
def dpo_loss_ref(
    log_prob_policy_win: float,
    log_prob_policy_lose: float,
    log_prob_ref_win: float,
    log_prob_ref_lose: float,
    beta: float = 0.1,
) -> float:
    reward_win = log_prob_policy_win - log_prob_ref_win
    reward_lose = log_prob_policy_lose - log_prob_ref_lose
    return -math.log(sigmoid(beta * (reward_win - reward_lose)) + 1e-8)


dpo_loss = dpo_loss_ref

# Test: policy strongly prefers winning response -> low loss
loss_good = dpo_loss(
    log_prob_policy_win=-1.0,    # policy assigns high prob to winner
    log_prob_policy_lose=-5.0,   # policy assigns low prob to loser
    log_prob_ref_win=-2.0,       # reference is more uncertain
    log_prob_ref_lose=-2.5,
)

loss_bad = dpo_loss(
    log_prob_policy_win=-5.0,    # policy wrongly prefers loser
    log_prob_policy_lose=-1.0,
    log_prob_ref_win=-2.0,
    log_prob_ref_lose=-2.5,
)

print(f'DPO loss (good policy): {loss_good:.4f}')  # should be low
print(f'DPO loss (bad policy):  {loss_bad:.4f}')   # should be high

---
## 6. Perplexity

**Perplexity (PPL)** = exponent of cross-entropy loss. Measures how 'surprised' the model is by the text.

**Formula:** `PPL = exp(-1/N * sum(log P(w_i | w_<i)))`

- Lower PPL = model assigns higher probability to the text = better fit
- PPL = 1: perfect (impossible in practice)
- PPL = vocab_size: random guessing
- GPT-2: PPL ~29 on PTB; GPT-4: PPL < 10

**Limitations:** PPL is sensitive to tokenization, so models with different tokenizers can't be compared fairly.

In [ ]:
def compute_perplexity(
    log_probs: np.ndarray  # (seq_len,) log probability of each actual token
) -> float:
    """
    Compute perplexity from per-token log probabilities.
    """
    # YOUR CODE HERE
    raise NotImplementedError()


# REFERENCE
def compute_perplexity_ref(log_probs: np.ndarray) -> float:
    # Average negative log likelihood
    avg_nll = -np.mean(log_probs)
    return float(np.exp(avg_nll))


compute_perplexity = compute_perplexity_ref

# Examples
# Perfect: always assigns prob 1.0 to correct token
perfect_log_probs = np.zeros(100)  # log(1) = 0
print('Perfect PPL:', compute_perplexity(perfect_log_probs))  # 1.0

# Random: assigns 1/32000 prob to each token
random_log_probs = np.full(100, math.log(1/32000))
print('Random PPL (vocab=32k):', compute_perplexity(random_log_probs))  # ~32000

# Good LLM: assigns ~10% prob to correct token on average
good_log_probs = np.full(100, math.log(0.1))
print('Good LLM PPL:', compute_perplexity(good_log_probs))  # 10

---
## 7. Grouped Query Attention (GQA)

**Standard MHA:** Q, K, V all have `num_heads` heads. KV cache = `num_heads * seq_len * head_dim * 2`.

**GQA:** Multiple Q heads share a single K/V head. Mistral 7B uses 32 Q heads but only 8 KV heads.

**Memory saving:** KV cache reduced by factor of `num_q_heads / num_kv_heads = 32/8 = 4x`.

**MQA (Multi-Query Attention):** Extreme case — 1 KV head shared by all Q heads.

In [ ]:
def gqa_attention(
    Q: np.ndarray,    # (batch, num_q_heads, seq, head_dim)
    K: np.ndarray,    # (batch, num_kv_heads, seq, head_dim)
    V: np.ndarray,    # (batch, num_kv_heads, seq, head_dim)
    mask: Optional[np.ndarray] = None,
) -> np.ndarray:
    """
    Grouped Query Attention.
    Each KV head is shared by (num_q_heads / num_kv_heads) Q heads.
    """
    batch, num_q_heads, seq_q, head_dim = Q.shape
    num_kv_heads = K.shape[1]
    
    assert num_q_heads % num_kv_heads == 0
    groups = num_q_heads // num_kv_heads
    
    # YOUR CODE HERE
    # Option 1: Repeat K and V to match num_q_heads, then do standard MHA
    # np.repeat(K, groups, axis=1) repeats each KV head 'groups' times
    raise NotImplementedError()


# REFERENCE
def gqa_attention_ref(
    Q: np.ndarray,
    K: np.ndarray,
    V: np.ndarray,
    mask: Optional[np.ndarray] = None,
) -> np.ndarray:
    batch, num_q_heads, seq_q, head_dim = Q.shape
    num_kv_heads = K.shape[1]
    groups = num_q_heads // num_kv_heads
    
    # Expand K, V to match Q heads (no copy — just repeat along head dim)
    K_expanded = np.repeat(K, groups, axis=1)  # (batch, num_q_heads, seq, head_dim)
    V_expanded = np.repeat(V, groups, axis=1)
    
    # Standard attention
    scores = np.einsum('bhqd,bhkd->bhqk', Q, K_expanded) / math.sqrt(head_dim)
    
    if mask is not None:
        scores = np.where(mask, -1e9, scores)
    
    weights = softmax(scores, axis=-1)
    return np.einsum('bhqk,bhkd->bhqd', weights, V_expanded)


gqa_attention = gqa_attention_ref

# Test: Mistral 7B config
batch, seq, num_q, num_kv, head_dim = 2, 8, 32, 8, 128
Q = np.random.randn(batch, num_q, seq, head_dim)
K = np.random.randn(batch, num_kv, seq, head_dim)
V = np.random.randn(batch, num_kv, seq, head_dim)

out = gqa_attention(Q, K, V)
print('GQA output shape:', out.shape)  # (2, 32, 8, 128)

# KV cache comparison
kv_mha = 2 * 32 * seq * head_dim * 2 / 1e6  # full MHA KV cache (MB)
kv_gqa = 2 * 8 * seq * head_dim * 2 / 1e6   # GQA KV cache (MB)
print(f'\nKV cache per layer: MHA={kv_mha:.2f}MB, GQA={kv_gqa:.2f}MB ({kv_gqa/kv_mha:.0%})')

---
## 8. Sliding Window Attention (SWA)

**Standard attention:** Each token attends to ALL previous tokens. O(N²) complexity.

**SWA:** Each token attends to only the last W tokens (window). O(N*W) complexity.

**Key insight:** Via multiple SWA layers, information still flows from distant tokens through the stack — each layer's effective receptive field grows by W tokens.

In [ ]:
def make_sliding_window_mask(seq_len: int, window_size: int) -> np.ndarray:
    """
    Create a sliding window causal mask.
    Token at position i can attend to positions [i-window_size+1, i].
    
    True = masked out (cannot attend).
    Returns: (seq_len, seq_len) boolean array.
    """
    # YOUR CODE HERE
    # Causal mask: upper triangle
    # Window mask: positions more than window_size steps back
    # Combined: mask = upper_triangle OR too_far_back
    raise NotImplementedError()


# REFERENCE
def make_sliding_window_mask_ref(seq_len: int, window_size: int) -> np.ndarray:
    causal = np.triu(np.ones((seq_len, seq_len), dtype=bool), k=1)  # future tokens
    
    # Position indices
    row_idx = np.arange(seq_len)[:, np.newaxis]  # (seq, 1)
    col_idx = np.arange(seq_len)[np.newaxis, :]  # (1, seq)
    outside_window = (row_idx - col_idx) >= window_size  # too far in the past
    
    return causal | outside_window


make_sliding_window_mask = make_sliding_window_mask_ref

mask = make_sliding_window_mask(8, window_size=3)
print('Sliding Window Mask (True=masked, window=3, seq=8):')
print('Row=query, Col=key')
print(mask.astype(int))
print('Each row: can attend to at most 3 tokens to the left + itself')

---
## 9. RoPE (Rotary Position Embedding)

**Key idea:** Instead of adding position vectors to embeddings, rotate Q and K vectors by an angle that depends on position.

**Why RoPE?**
- Encodes *relative* position directly in the attention score
- Naturally extrapolates to longer sequences than seen in training
- No extra parameters

**Formula:** `RoPE(x, pos) = x ⊗ cos(θ) + rotate_half(x) ⊗ sin(θ)`

where θ depends on position and dimension index.

In [ ]:
def precompute_rope_frequencies(head_dim: int, max_seq_len: int, theta: float = 10000.0) -> Tuple[np.ndarray, np.ndarray]:
    """
    Precompute cos and sin frequencies for RoPE.
    
    Returns:
        cos_freq: (max_seq_len, head_dim)
        sin_freq: (max_seq_len, head_dim)
    """
    # Frequency for each pair of dimensions
    # theta_i = 1 / (theta ^ (2i / head_dim)) for i in range(head_dim // 2)
    i = np.arange(0, head_dim, 2, dtype=float)
    freqs = 1.0 / (theta ** (i / head_dim))  # (head_dim/2,)
    
    positions = np.arange(max_seq_len)  # (max_seq_len,)
    angles = np.outer(positions, freqs)  # (max_seq_len, head_dim/2)
    
    # Repeat each angle twice (for x1, x2 pairs)
    angles = np.concatenate([angles, angles], axis=-1)  # (max_seq_len, head_dim)
    
    return np.cos(angles), np.sin(angles)


def apply_rope(
    x: np.ndarray,           # (batch, num_heads, seq, head_dim)
    cos_freq: np.ndarray,    # (seq, head_dim)
    sin_freq: np.ndarray,    # (seq, head_dim)
) -> np.ndarray:
    """
    Apply RoPE to query or key vectors.
    """
    # YOUR CODE HERE
    # rotate_half(x): swap and negate pairs: [-x2, -x4, ...] with [x1, x3, ...]
    # result = x * cos + rotate_half(x) * sin
    raise NotImplementedError()


# REFERENCE
def rotate_half(x: np.ndarray) -> np.ndarray:
    """Rotate half: [..., x1, x2, ...] -> [..., -x2, x1, ...]"""
    half = x.shape[-1] // 2
    x1 = x[..., :half]
    x2 = x[..., half:]
    return np.concatenate([-x2, x1], axis=-1)


def apply_rope_ref(
    x: np.ndarray,
    cos_freq: np.ndarray,
    sin_freq: np.ndarray,
) -> np.ndarray:
    # cos_freq, sin_freq: (seq, head_dim) -> add batch and heads dims
    cos = cos_freq[np.newaxis, np.newaxis, :, :]  # (1, 1, seq, head_dim)
    sin = sin_freq[np.newaxis, np.newaxis, :, :]
    return x * cos + rotate_half(x) * sin


apply_rope = apply_rope_ref

cos_f, sin_f = precompute_rope_frequencies(head_dim=64, max_seq_len=512)
x = np.random.randn(2, 8, 16, 64)  # batch, heads, seq, head_dim
x_roped = apply_rope(x, cos_f[:16], sin_f[:16])
print('RoPE output shape:', x_roped.shape)

# Key property: relative position is preserved in dot product
# Verify: dot product of RoPE(q, pos1) and RoPE(k, pos2) depends only on (pos1 - pos2)
q = np.random.randn(1, 1, 1, 64)
k = np.random.randn(1, 1, 1, 64)

# Same relative position (both shift by 5)
q_pos3 = apply_rope(q, cos_f[3:4], sin_f[3:4])
k_pos1 = apply_rope(k, cos_f[1:2], sin_f[1:2])
q_pos8 = apply_rope(q, cos_f[8:9], sin_f[8:9])
k_pos6 = apply_rope(k, cos_f[6:7], sin_f[6:7])

dot1 = np.sum(q_pos3 * k_pos1)  # relative pos = 2
dot2 = np.sum(q_pos8 * k_pos6)  # relative pos = 2
print(f'\nDot products (same relative pos=2):')
print(f'  pos(3,1): {dot1:.4f}')
print(f'  pos(8,6): {dot2:.4f}')
print(f'  Are they equal? {np.isclose(dot1, dot2)}')

---
## 10. Final Practice: Spot the Bug

Each snippet has a subtle bug. Find it without running the code.

In [ ]:
# Bug 1: Attention
def buggy_attention(Q, K, V):
    d_k = Q.shape[-1]
    scores = Q @ K.T  # BUG: what's wrong here?
    weights = softmax(scores / math.sqrt(d_k))
    return weights @ V
# ANSWER: For batched input (batch, seq, d_k), K.T transposes wrong dims
# Should be K.transpose(0, 2, 1) to get (batch, d_k, seq)


# Bug 2: Top-p sampling
def buggy_top_p(logits, p):
    probs = softmax(logits)
    sorted_indices = np.argsort(probs)  # BUG: what's wrong?
    sorted_probs = probs[sorted_indices]
    cumulative = np.cumsum(sorted_probs)
    cutoff = np.searchsorted(cumulative, p)
    nucleus = sorted_indices[:cutoff]
    return int(np.random.choice(nucleus, p=probs[nucleus]/probs[nucleus].sum()))
# ANSWER: argsort sorts ascending; we need DESCENDING for top-p (highest prob first)
# Fix: np.argsort(probs)[::-1]


# Bug 3: BPE
def buggy_bpe_merge(pair, word_freqs):
    merged = pair[0] + pair[1]
    new_word_freqs = {}
    for word_tuple, freq in word_freqs.items():
        new_word = list(word_tuple)
        i = 0
        while i < len(new_word) - 1:
            if new_word[i] == pair[0] and new_word[i+1] == pair[1]:
                new_word[i] = merged
                del new_word[i+1]  # BUG: what's wrong here?
                i += 1
            else:
                i += 1
        new_word_freqs[tuple(new_word)] = freq
    return new_word_freqs
# ANSWER: After merging, we should NOT increment i (the merged token might pair with next)
# Standard fix: don't increment i after a merge — check the new merged token against next


# Bug 4: LoRA
def buggy_lora_init(in_feat, out_feat, rank):
    A = np.zeros((in_feat, rank))        # BUG: what's wrong?
    B = np.random.randn(rank, out_feat) * 0.02
    return A, B
# ANSWER: A should be randomly initialized, B should be ZERO
# Reason: B=0 ensures ΔW = A@B = 0 at init, so model starts as the base model
# If A=0 and B is random, same thing happens, BUT A=0 means no gradient flows through A!

print('Check the code comments for bug explanations.')

---
## 11. Interview Questions (Advanced)

1. **MoE:** Why does Mixtral use exactly 2 of 8 experts? What happens if you use 1? Or 4?

2. **Load balancing:** Why does expert collapse happen? How does the auxiliary loss prevent it?

3. **LoRA rank:** A customer wants to fine-tune Mistral 7B on their company's Q&A data. What rank would you recommend? What alpha?

4. **DPO vs RLHF:** Why might DPO underperform RLHF on complex tasks? (hint: reward hacking, distribution mismatch)

5. **GQA:** Mixtral uses 8 KV heads but 32 Q heads. During generation, what's the KV cache memory for a batch of 16 at seq_len 8192?

6. **RoPE vs sinusoidal:** If a model trained with RoPE at max_len=4096 needs to handle 32k context, what modifications enable this? (YaRN, NTK-aware scaling)

---
## You're Ready. Final Tips:

1. **Think out loud** — the interviewer wants to hear your reasoning, not just the answer
2. **Ask clarifying questions** — 'Do you want me to implement from scratch or use NumPy?'
3. **Write tests first** — shows engineering maturity
4. **Name your variables clearly** — `d_model`, `num_heads`, `head_dim` not `x`, `y`, `z`
5. **Add type hints** — Mistral loves typed Python
6. **Know the WHY** — not just the formula, but why each design choice was made

Good luck Yash!